# 🔍 EVTX Timeline Analyzer
### Automated Attack Chain Detection for SOC & Forensics

**What this tool does:**
- 📂 Upload any `.evtx` file (Security, Sysmon, PowerShell, etc.)
- 🧠 Automatically detects attack patterns (Brute Force, Mimikatz, etc.)
- 📊 Generates human-readable timeline reports
- 🎨 Visualizes attack chains as interactive graphs
- 📤 Exports JSON data for further analysis

---

## ⚙️ Step 1: Setup & Dependencies

In [ ]:
# Install required libraries
!pip install python-evtx pandas networkx matplotlib -q

import os
import sys
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Clone or update the repository
REPO_NAME = "EVTX-TiMLiner"
REPO_URL = "https://github.com/JoeSherif97/EVTX-TiMLiner.git"

if not os.path.exists(f"/content/{REPO_NAME}"):
    !git clone {REPO_URL}
else:
    %cd {REPO_NAME}
    !git pull
    %cd ..

sys.path.append(f"/content/{REPO_NAME}")

from parser_universal import parse_evtx_to_df, map_to_common_fields
from analyze_evtx import generate_timeline_report

print("✅ All dependencies installed and ready!")

## 📤 Step 2: Upload Your EVTX File

Click the button below to upload your `.evtx` file.

In [ ]:
from google.colab import files

print("📤 Please select an .evtx file to analyze...")
uploaded = files.upload()

if uploaded:
    filename = list(uploaded.keys())[0]
    print(f"✅ Uploaded: {filename}")
    print(f"📊 File size: {os.path.getsize(filename):,} bytes")
else:
    print("❌ No file uploaded. Please upload an .evtx file.")

## 🧠 Step 3: Run Attack Chain Analysis

This will parse the file and generate a complete timeline report.

In [ ]:
# Get the uploaded file
filename = list(uploaded.keys())[0]

# Parse the file
print(f"🔍 Parsing: {filename}")
df = parse_evtx_to_df(filename)

if len(df) == 0:
    print("❌ No events parsed from this file. Please check if it's a valid EVTX.")
else:
    # Map to common fields
    df_mapped = map_to_common_fields(df)
    
    # Generate report
    from analyze_evtx import generate_timeline_report
    report_text = generate_timeline_report(df_mapped, filename)
    
    print("\n" + report_text)
    
    # Save report
    base_name = os.path.splitext(filename)[0]
    report_file = f"{base_name}_report.txt"
    with open(report_file, 'w') as f:
        f.write(report_text)
    print(f"\n✅ Report saved as: {report_file}")

## 🎨 Step 4: Generate Attack Chain Visualization

In [ ]:
from visualize_chain import create_attack_graph

base_name = os.path.splitext(filename)[0]
graph_file = f"{base_name}_chain.png"

print("🎨 Creating attack chain visualization...")
G = create_attack_graph(df_mapped, filename, graph_file)

if G:
    print(f"✅ Graph saved as: {graph_file}")
    
    # Display the image
    from IPython.display import Image, display
    display(Image(graph_file))
else:
    print("⚠️ Could not generate graph (not enough events)")

## 📤 Step 5: Export Data (Optional)

Export the parsed data as JSON for further analysis.

In [ ]:
import json

base_name = os.path.splitext(filename)[0]
json_file = f"{base_name}_data.json"

# Convert DataFrame to JSON
json_data = df_mapped.to_dict(orient='records')
for record in json_data:
    for key, value in record.items():
        if hasattr(value, 'isoformat'):
            record[key] = value.isoformat()
        elif isinstance(value, pd.Timestamp):
            record[key] = value.isoformat()

with open(json_file, 'w') as f:
    json.dump(json_data, f, indent=2, default=str)

print(f"✅ Data exported as: {json_file}")
print(f"📊 Total events exported: {len(json_data)}")

from google.colab import files
files.download(json_file)

## 📊 Step 6: Summary Dashboard

In [ ]:
if len(df) > 0:
    print("="*60)
    print("📊 ANALYSIS SUMMARY")
    print("="*60)
    
    print(f"\n📁 File: {filename}")
    print(f"📊 Total Events: {len(df)}")
    
    if 'timestamp' in df.columns and not df['timestamp'].isna().all():
        print(f"📅 Time Range: {df['timestamp'].min()} to {df['timestamp'].max()}")
    
    # Event breakdown
    if 'event_id' in df.columns:
        print("\n📊 Event Breakdown:")
        event_counts = df['event_id'].value_counts()
        event_names = {
            '4624': '✅ Successful Logon',
            '4625': '❌ Failed Logon',
            '4688': '💻 Process Creation',
            '3': '🌐 Network Connection',
            '10': '🔐 Process Access',
            '1': '💻 Process Creation',
            '4104': '📜 PowerShell Script'
        }
        for eid, count in event_counts.items():
            name = event_names.get(str(eid), f'Event {eid}')
            print(f"  - {eid}: {count} ({name})")
    
    # Users
    if 'username' in df.columns:
        users = df['username'].dropna().unique()
        if len(users) > 0:
            print(f"\n👤 Users Found: {', '.join(users[:5])}")
            if len(users) > 5:
                print(f"   (and {len(users)-5} more...)")
    
    # Attack detection
    print("\n🔍 Attack Detection:")
    
    # Check for brute force
    failed_count = len(df[df['event_id'] == '4625'])
    success_count = len(df[df['event_id'] == '4624'])
    if failed_count > 0 and success_count > 0:
        print("  🚨 BRUTE FORCE PATTERN DETECTED!")
        print(f"     {failed_count} failures → {success_count} successes")
    elif failed_count > 0:
        print("  ⚠️ Brute force attempt detected (no success)")
    
    # Check for Mimikatz
    process_access = df[df['event_id'] == '10']
    if not process_access.empty:
        for _, row in process_access.iterrows():
            target = str(row.get('TargetImage', ''))
            if 'lsass' in target.lower():
                print("  🚨 MIMIKATZ DETECTED! (lsass.exe access)")
                break
    
    # Check for network connections
    network_events = df[df['event_id'] == '3']
    if not network_events.empty:
        print(f"  🌐 {len(network_events)} network connections detected")
    
    print("\n" + "="*60)
    print("✅ Analysis complete!")
    print("="*60)
else:
    print("❌ No data to summarize")

## 📥 Step 7: Download Results

Click the links below to download all generated files.

In [ ]:
base_name = os.path.splitext(filename)[0]
files_to_download = [
    f"{base_name}_report.txt",
    f"{base_name}_chain.png",
    f"{base_name}_data.json"
]

print("📥 Download available files:")
for f in files_to_download:
    if os.path.exists(f):
        size = os.path.getsize(f)
        print(f"  ✅ {f} ({size:,} bytes)")
    else:
        print(f"  ❌ {f} (not found)")

print("\n📥 To download, right-click the file in the file browser (left sidebar)")
print("   or use the code below to download specific files:")

# Optional: Download specific file
# from google.colab import files
# files.download(f"{base_name}_report.txt")

---

## 🎯 Summary

You've successfully analyzed an EVTX file using the universal parser.

**Generated Files:**
- 📄 `{filename}_report.txt` - Human-readable timeline report
- 🎨 `{filename}_chain.png` - Attack chain visualization
- 📊 `{filename}_data.json` - Full parsed data in JSON

**Support:**
- 📂 Works with ANY EVTX file (Security, Sysmon, PowerShell, etc.)
- 🔍 Auto-detects brute force, Mimikatz, and other attack patterns
- 🚀 Ready for SOC and forensics workflows

---